# 고객 행동 데이터 기반 퀵커머스 개인화 서비스 개선

- 회원·상품·주문 데이터를 기반으로 수행된 데이터 분석 프로젝트
- 데이터 보안 이슈로 원본 데이터는 포함하지 않음
- 분석 로직 및 핵심 결과 재현 중심으로 구성

## 분석 목표
- 구독 전환 요인 분석
- 고객 구매 패턴 기반 개인화 전략 도출
- 휴면 고객 정의 및 리텐션 전략 수립

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

sns.set_style("whitegrid")

In [ ]:
df = pd.read_csv("../data/sales_data.csv")

grouped = df.groupby("is_subscribed").agg({
    "purchase_amount": "mean",
    "purchase_count": "mean"
})

display(grouped)

In [ ]:
sub = df[df["is_subscribed"] == 1]["monthly_freq"]
non_sub = df[df["is_subscribed"] == 0]["monthly_freq"]

t_stat, p_value = ttest_ind(sub, non_sub, equal_var=False)

print(f"p-value: {p_value:.4f}")

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x="is_subscribed", y="monthly_freq", data=df)
plt.title("구독 여부별 구매 빈도 비교")
plt.show()

In [ ]:
from mlxtend.frequent_patterns import fpgrowth, association_rules

basket = (
    df.groupby(["order_id", "product"])["product"]
    .count().unstack().fillna(0)
)

basket = basket.applymap(lambda x: 1 if x > 0 else 0)

freq_items = fpgrowth(basket, min_support=0.005, use_colnames=True)

rules = association_rules(freq_items, metric="lift", min_threshold=1.0)

rules = rules.sort_values("lift", ascending=False)

display(rules.head(10))

In [ ]:
snapshot_date = df["order_date"].max()

rfm = df.groupby("customer_id").agg({
    "order_date": lambda x: (snapshot_date - x.max()).days,
    "order_id": "count",
    "purchase_amount": "sum"
})

rfm.columns = ["Recency", "Frequency", "Monetary"]

display(rfm.head())

In [ ]:
rfm["is_dormant"] = np.where(rfm["Recency"] >= 60, 1, 0)

print("휴면 비율:", rfm["is_dormant"].mean())

In [ ]:
merged = rfm.merge(
    df[["customer_id", "is_subscribed"]].drop_duplicates(),
    on="customer_id"
)

dormant_ratio = merged.groupby("is_dormant")["is_subscribed"].mean()
display(dormant_ratio)

In [ ]:
def risk_group(freq):
    if freq <= 2:
        return "High"
    elif freq <= 5:
        return "Mid"
    else:
        return "Low"

rfm["risk_group"] = rfm["Frequency"].apply(risk_group)

display(rfm["risk_group"].value_counts())

In [ ]:
age_df = df[["customer_id", "age"]].drop_duplicates()

rfm = rfm.merge(age_df, on="customer_id")

rfm["age_group"] = pd.cut(
    rfm["age"],
    bins=[0,30,50,100],
    labels=["20-30", "40-50", "60+"]
)

display(rfm.groupby("age_group")["is_dormant"].mean())

In [ ]:
## 결론

- 구독 전환은 소비 금액보다 구매 빈도(리듬)에 영향
- 연관 분석을 통해 묶음 상품 및 추천 전략 가능
- 전체 고객 중 약 37%가 휴면 상태
- 휴면 고객은 비구독자 비율이 높아 리텐션 전략 핵심 타겟
- High/Mid 위험군 중심 전략이 ROI 측면에서 효과적